# Training YOLO Fire & Smoke di Google Colab

Notebook ini melatih model dari dataset Kaggle yang sudah tersimpan di Google Drive. Dataset berformat YOLO dengan kelas `0=smoke`, `1=fire`.

1. Buka notebook di Google Colab, pilih **Runtime → Change runtime type → T4 GPU**.
2. Atur `DATASET_SOURCE` di cell konfigurasi ke folder dataset yang sudah diekstrak atau file ZIP di Drive.
3. Jalankan semua cell. `best.pt` disimpan ke Drive dan diunduh ke komputer.
4. Salin `best.pt` ke `fire-smoke-detection/models/fire-smoke-yolov8n/best.pt`. Media Lab frontend mengirim gambar ke FastAPI; Live Scan webcam saat ini tetap memakai pemindai lokal browser.

Dataset: [Smoke-Fire-Detection-YOLO](https://www.kaggle.com/datasets/sayedgamal99/smoke-fire-detection-yolo/data) · lisensi dataset CC0. Model awal: YOLOv8n.

In [ ]:
%pip -q install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, zipfile
import torch, yaml
from ultralytics import YOLO

# Ganti ke folder atau ZIP dataset yang sudah Anda simpan di Google Drive.
DATASET_SOURCE = Path('/content/drive/MyDrive/datasets/smoke-fire-detection-yolo')
# Pengaturan training dapat disesuaikan.
BASE_MODEL = 'yolov8n.pt'
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16

WORK_DIR = Path('/content/smoke-fire-dataset')
RUNS_DIR = Path('/content/runs/fire-smoke')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/fire-smoke-detection/training-output/fire-smoke-yolov8n')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'tidak tersedia (CPU lebih lambat)')
print('Sumber dataset:', DATASET_SOURCE)

In [ ]:
# Salin dataset ke disk lokal Colab (lebih cepat daripada training langsung dari Drive).
if not DATASET_SOURCE.exists():
    raise FileNotFoundError(f'Dataset tidak ditemukan: {DATASET_SOURCE}. Periksa DATASET_SOURCE.')

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
if DATASET_SOURCE.is_file() and DATASET_SOURCE.suffix.lower() == '.zip':
    with zipfile.ZipFile(DATASET_SOURCE) as archive:
        root = WORK_DIR.resolve()
        if any(not (WORK_DIR / item.filename).resolve().is_relative_to(root) for item in archive.infolist()):
            raise ValueError('ZIP berisi path di luar folder ekstraksi.')
        archive.extractall(WORK_DIR)
else:
    shutil.copytree(DATASET_SOURCE, WORK_DIR, dirs_exist_ok=True)

# Dataset Kaggle bisa punya satu folder pembungkus. Cari struktur train/images.
candidates = [WORK_DIR, *WORK_DIR.glob('*/'), *WORK_DIR.glob('*/*/')]
DATASET_DIR = next((p for p in candidates if (p / 'train/images').is_dir()), None)
if DATASET_DIR is None:
    raise FileNotFoundError('Struktur train/images tidak ditemukan. Pastikan dataset YOLO sudah benar.')
VAL_SPLIT = next((name for name in ('val', 'valid') if (DATASET_DIR / name / 'images').is_dir()), None)
if VAL_SPLIT is None or not (DATASET_DIR / 'train/labels').is_dir() or not (DATASET_DIR / VAL_SPLIT / 'labels').is_dir():
    raise FileNotFoundError('Folder train/labels dan val/labels (atau valid/labels) wajib tersedia.')

# Tulis YAML sendiri agar path sesuai lokasi runtime Colab dan urutan kelas sesuai dataset Kaggle.
data_config = {
    'path': str(DATASET_DIR),
    'train': 'train/images',
    'val': f'{VAL_SPLIT}/images',
    'names': {0: 'smoke', 1: 'fire'},
}
if (DATASET_DIR / 'test/images').is_dir():
    data_config['test'] = 'test/images'
DATA_YAML = Path('/content/fire-smoke-data.yaml')
DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding='utf-8')
print(DATA_YAML.read_text())
print('Jumlah gambar train:', len(list((DATASET_DIR / 'train/images').glob('*'))))
print('Jumlah gambar validation:', len(list((DATASET_DIR / VAL_SPLIT / 'images').glob('*'))))

In [ ]:
model = YOLO(BASE_MODEL)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=2,
    project=str(RUNS_DIR),
    name='fire-smoke-yolov8n',
    exist_ok=True,
    patience=15,
    seed=42,
)
RUN_DIR = RUNS_DIR / 'fire-smoke-yolov8n'
BEST_PT = RUN_DIR / 'weights/best.pt'
assert BEST_PT.is_file(), f'Bobot best.pt tidak ditemukan: {BEST_PT}'
print('Best weights:', BEST_PT)

In [ ]:
# Evaluasi pada test split bila tersedia.
if 'test' in data_config:
    best_model = YOLO(str(BEST_PT))
    metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=IMAGE_SIZE, device=0 if torch.cuda.is_available() else 'cpu')
    print(metrics.results_dict)
else:
    print('Split test tidak ditemukan; evaluasi dilewati.')

# Simpan bobot dan ringkasan training ke Google Drive.
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
drive_best = DRIVE_OUTPUT / 'best.pt'
shutil.copy2(BEST_PT, drive_best)
for filename in ('results.csv', 'args.yaml', 'confusion_matrix.png'):
    artifact = RUN_DIR / filename
    if artifact.is_file():
        shutil.copy2(artifact, DRIVE_OUTPUT / filename)
print('Bobot tersimpan di Drive:', drive_best)

In [ ]:
# Unduh best.pt, lalu salin ke folder models pada proyek lokal.
from google.colab import files
files.download(str(drive_best))

## Pakai hasil training di aplikasi

Salin file yang terunduh ke `fire-smoke-detection/models/fire-smoke-yolov8n/best.pt` (timpa file lama bila ada). Jalankan backend dari folder `backend`:

```bash
pip install -r requirements-yolo.txt
uvicorn app.main:app --reload --host 0.0.0.0 --port 8000
```

Pastikan `frontend/.env.local` berisi `NEXT_PUBLIC_BACKEND_URL=http://localhost:8000`. Cek `GET http://localhost:8000/api/v1/health`; field `engine` harus menunjukkan `yolo:fire-smoke-yolov8n/best.pt`. Media Lab memakai model melalui FastAPI. Live Scan webcam yang ada saat ini masih menjalankan pemindai lokal browser.